
1.Builds rain_features.csv, grid_terrain.csv and training_table.csv.
2.Run top to bottom. Raw inputs live in data/raw/, labels in data/labels/.
3.Rainfall was downloaded with timezone=Asia/Kolkata (confirmed in both requests in notebook 01).

In [2]:
import pandas as pd, numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"

rain feature


In [3]:
r = pd.read_csv(RAW / "rainfall_2022_2026.csv", parse_dates=["time"]).set_index("time")
r = r.asfreq("h")                                   # force a strict hourly index
assert r["precipitation"].isna().sum() == 0, "missing hours in rainfall data"

rain = pd.DataFrame(index=r.index)
rain["rain_1h"] = r["precipitation"]
for h in [3, 6, 24, 72]:
    rain[f"rain_{h}h"] = r["precipitation"].rolling(h, min_periods=h).sum()

rain.to_csv(PROC / "rain_features.csv")
print("rain_features:", rain.shape)

rain_features: (41880, 5)


In [4]:
# storm finder
rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
daily = rain["rain_1h"].resample("D").sum()
peak_hr = rain["rain_1h"].groupby(rain.index.date).idxmax()

top = daily.sort_values(ascending=False).head(15).round(1).to_frame("day_total_mm")
top["peak_hour"] = [peak_hr[d.date()].strftime("%H:%M") for d in top.index]
print(top.sort_index())



            day_total_mm peak_hour
time                              
2022-06-30          47.7     05:00
2022-09-14          40.2     03:00
2022-09-15          82.5     21:00
2023-07-01          39.4     18:00
2024-07-01          58.6     23:00
2024-07-02          80.0     07:00
2024-07-05          44.2     17:00
2024-08-09          40.7     13:00
2024-09-11          57.7     15:00
2024-09-12          57.9     20:00
2025-06-30          38.7     14:00
2025-08-03          40.8     07:00
2025-08-04          66.8     03:00
2026-09-25          93.8     20:00
2026-09-26          41.1     03:00


In [5]:
# terrain
e = pd.read_csv(RAW / "lucknow_grid_elevation.csv")
coords, elev = e[["lat", "lon"]].to_numpy(), e["elevation"].to_numpy()
dist = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1))

mask = (dist > 0) & (dist <= 0.025)          # neighbours within ~2.5 km, self excluded
e["relative_elev"] = elev - (mask * elev[None, :]).sum(1) / mask.sum(1)

e.to_csv(PROC / "grid_terrain.csv", index=False)
print(e["relative_elev"].describe().round(2))

count    483.00
mean       0.04
std        2.82
min      -12.15
25%       -1.57
50%       -0.25
75%        1.70
max       12.93
Name: relative_elev, dtype: float64


In [6]:
# training table
rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
grid = pd.read_csv(PROC / "grid_terrain.csv")
labels = pd.read_csv(ROOT / "data" / "labels" / "waterlogging_reports.csv", parse_dates=["date"])
print("labels loaded:", len(labels))

labels["label"] = (labels["severity"].str.lower() != "none").astype(int)

rows = []
for _, r in labels.iterrows():
    day = rain[rain.index.strftime("%Y-%m-%d") == r["date"].strftime("%Y-%m-%d")]
    if day.empty:
        print("No rain data for", r["date"].date()); continue
    hour = day["rain_1h"].idxmax()

    d = (grid["lat"] - r["lat"]) ** 2 + (grid["lon"] - r["lon"]) ** 2
    if np.sqrt(d.min()) > 0.01:
        print("Skipping (too far from grid):", r["location_name"]); continue
    t = grid.loc[d.idxmin()]

    rows.append({
        "location_name": r["location_name"], "event_hour": hour, "label": r["label"],
        **rain.loc[hour, ["rain_1h", "rain_3h", "rain_6h", "rain_24h", "rain_72h"]].to_dict(),
        "elevation": t["elevation"], "relative_elev": t["relative_elev"],
    })

table = pd.DataFrame(rows)
table.to_csv(PROC / "training_table.csv", index=False)
print(table.round({"rain_1h": 1, "rain_3h": 1, "rain_6h": 1, "rain_24h": 1, "rain_72h": 1,
                   "elevation": 0, "relative_elev": 1}).sort_values(["event_hour", "location_name"]))
print("\nLabel counts:\n", table["label"].value_counts())



labels loaded: 56
Skipping (too far from grid): Sarojini Nagar
                       location_name          event_hour  label  rain_1h  \
17                          Alambagh 2022-07-21 03:00:00      0      6.9   
13                           Aliganj 2022-07-21 03:00:00      0      6.9   
18                          Ashiyana 2022-07-21 03:00:00      0      6.9   
14                          Charbagh 2022-07-21 03:00:00      0      6.9   
15                             Chowk 2022-07-21 03:00:00      0      6.9   
19                      Faizullaganj 2022-07-21 03:00:00      0      6.9   
12                       Gomti Nagar 2022-07-21 03:00:00      0      6.9   
11                        Hazratganj 2022-07-21 03:00:00      0      6.9   
16                      Indira Nagar 2022-07-21 03:00:00      0      6.9   
44                          Alambagh 2022-07-30 02:00:00      0      6.7   
40                           Aliganj 2022-07-30 02:00:00      0      6.7   
45                       

In [7]:
# do the rain feature seprate the classes
print(table.groupby("label")[["rain_1h", "rain_24h", "rain_72h"]].agg(["min", "mean", "max"]).round(1))


      rain_1h             rain_24h             rain_72h              
          min  mean   max      min  mean   max      min   mean    max
label                                                                
0         4.1   6.0   6.9     11.7  19.3  31.4     19.0   25.6   32.2
1         2.5  10.6  13.0     32.2  65.8  76.4     83.6  110.1  137.7


In [8]:
# negative candidates
daily = rain["rain_1h"].resample("D").sum()
max72 = rain["rain_72h"].groupby(rain.index.date).max()
heavy = daily[daily >= 35].index
labelled = pd.to_datetime(labels["date"].unique())

near = set()
for day0 in list(heavy) + list(labelled):
    for k in range(-3, 4):
        near.add(day0 + pd.Timedelta(days=k))

cand = daily[(daily >= 8) & (daily <= 30) & (~daily.index.isin(near))]
cand = cand[[max72[d.date()] <= 50 for d in cand.index]]

peak = rain["rain_1h"].groupby(rain.index.date).max()
out = cand.sample(min(10, len(cand)), random_state=0).sort_index().round(1).to_frame("day_total_mm")
out["max_1h"] = [round(peak[d.date()], 1) for d in out.index]
out["max_72h"] = [round(max72[d.date()], 1) for d in out.index]
print(len(cand), "candidate days")
print(out)

daily.loc["2022-09-13":"2022-09-17"]



117 candidate days
            day_total_mm  max_1h  max_72h
time                                     
2022-08-03          15.8     4.4     24.9
2022-08-19          14.2     7.0     15.3
2023-05-26          12.5     5.2     16.0
2023-06-26          18.2     5.4     40.1
2024-08-19          12.7     5.4     27.1
2026-01-28          11.0     3.1     11.2
2026-06-13          23.4     6.9     33.1
2026-07-03           8.7     2.9     28.5
2026-07-07          28.1     8.1     32.4
2026-09-01          16.8     3.1     37.3


time
2022-09-13    24.7
2022-09-14    40.2
2022-09-15    82.5
2022-09-16    25.1
2022-09-17     6.1
Freq: D, Name: rain_1h, dtype: float64

In [9]:
import pandas as pd
dates = pd.to_datetime(["2022-06-30", "2023-07-01", "2024-07-02", "2024-09-12", "2025-08-04"])
print(daily.reindex(dates))

2022-06-30    47.7
2023-07-01    39.4
2024-07-02    80.0
2024-09-12    57.9
2025-08-04    66.8
Name: rain_1h, dtype: float64
